# Gomoku AlphaZero 학습 (Colab Pro)

9x9 보드, 5목 자유형 오목을 MCTS + 신경망 셀프플레이(AlphaZero 방식)로 학습합니다.

**사전 준비**
1. 로컬 `GomokuAI` 프로젝트 폴더 전체를 구글 드라이브에 업로드하세요 (예: `MyDrive/GomokuAI`).
2. Colab 상단 메뉴 `런타임 > 런타임 유형 변경`에서 GPU(T4/A100 등)를 선택하세요.
3. 매 세션 시작 시 이 노트북을 위에서부터 다시 실행하면 됩니다. 체크포인트가 드라이브에 저장되므로 세션이 끊겨도 마지막 iteration부터 자동 재개됩니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import sys

# 드라이브에 업로드한 GomokuAI 폴더 경로. 다른 위치에 두었다면 수정하세요.
PROJECT_DIR = '/content/drive/MyDrive/GomokuAI'

if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

import os
assert os.path.isdir(PROJECT_DIR), f"{PROJECT_DIR} 가 없습니다. 경로를 확인하세요."

In [ ]:
import torch
print('torch', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0))
else:
    print('GPU가 잡히지 않았습니다. 런타임 유형을 GPU로 변경하세요.')

## 설정
처음에는 파이프라인이 제대로 돌아가는지만 빠르게 확인하는 `SMOKE_TEST=True` 설정으로 몇 iteration만 돌려보고, 문제가 없으면 `SMOKE_TEST=False`로 바꿔 본격 학습을 시작하세요.

In [ ]:
from alphazero.config import AlphaZeroConfig

SMOKE_TEST = True

config = AlphaZeroConfig(
    board_size=9,
    win_length=5,
    checkpoint_dir=f'{PROJECT_DIR}/checkpoints',
)

if SMOKE_TEST:
    config.num_simulations = 16
    config.games_per_iteration = 8
    config.parallel_games = 8
    config.train_steps_per_iteration = 20
    config.arena_games = 4
    config.total_iterations = 3

config

In [ ]:
from alphazero.pipeline import run_training_loop

run_training_loop(config, resume=True)

## 학습 과정 확인
`checkpoints/log.csv` 에 iteration별 policy/value loss와 arena_win_rate(후보 모델이 현재 best 모델을 이기는 비율)가 쌓입니다. `promoted=True`가 한 번이라도 나오면 best 모델이 갱신된 것입니다.

In [ ]:
import pandas as pd

log_path = f'{config.checkpoint_dir}/log.csv'
df = pd.read_csv(log_path)
df.tail(20)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(df['iteration'], df['policy_loss']); axes[0].set_title('policy loss')
axes[1].plot(df['iteration'], df['value_loss']); axes[1].set_title('value loss')
axes[2].plot(df['iteration'], df['arena_win_rate']); axes[2].axhline(config.arena_win_rate_threshold, color='r', linestyle='--')
axes[2].set_title('arena win rate vs. current best')
plt.show()

## 현재 best 모델로 게임 한 판 구경하기 (애니메이션)
best.pt를 불러와 자기 자신과 대국시키고, 실제 수순을 오목판 애니메이션으로 재생합니다. 돌이 의미 있는 모양(연결/방어)을 만들면 학습이 진행되고 있다는 신호입니다.

In [ ]:
import numpy as np
from alphazero.arena import play_game
from alphazero.evaluator import NetEvaluator
from alphazero.model import GomokuNet

device = 'cuda' if torch.cuda.is_available() else 'cpu'
best_model = GomokuNet(config).to(device)
best_model.load_state_dict(torch.load(f'{config.checkpoint_dir}/best.pt', map_location=device))
evaluator = NetEvaluator(best_model, device)

rng = np.random.default_rng()
result, moves = play_game(evaluator, evaluator, config, rng, a_is_black=True, record_moves=True)
print('result:', result, '(a=흑 승, b=백 승, draw=무승부) |', len(moves), '수')

In [ ]:
from IPython.display import HTML
from gomoku_env.visualize import animate_game

anim = animate_game(moves, config.board_size, config.win_length, interval_ms=400)
HTML(anim.to_jshtml())